In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [9]:
import os

os.getcwd()

'C:\\Users\\Hp'

In [13]:

df = pd.read_csv(
    r"C:\Users\Hp\Desktop\world population.csv"
)

df.head()

,Country Name,Country Code,1960,1961,1962,1963,1964,1965,1966,1967,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Aruba,ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.012520,0.018612,0.023919,0.022253,0.018424,0.021701,0.029999,0.034214,NaN,NaN
1,Africa Eastern and Southern,AFE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.015093,12.886468,12.462475,11.991748,12.953763,14.734153,13.728741,14.185523,15.333073,15.265258
2,Afghanistan,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.634323,25.740314,26.420199,22.042897,25.773971,29.975583,33.597619,33.701432,34.743247,NaN
3,Africa Western and Central,AFW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.178587,20.596469,20.356769,20.549536,23.465495,25.398178,25.622994,25.371747,24.152889,22.555409
4,Angola,AGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.663772,14.671141,14.928326,13.339791,12.391159,15.022367,16.561924,18.989093,20.689762,22.069326


In [15]:
df.columns

Index(['Country Name', 'Country Code', '1960', '1961', '1962', '1963', '1964',
       '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973',
       '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982',
       '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991',
       '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000',
       '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009',
       '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018',
       '2019', '2020', '2021', '2022', '2023', '2024'],
      dtype='object')

In [19]:
print(df.columns.tolist())

['Country Name', 'Country Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


In [21]:
# Convert wide data to long format

year_cols = [col for col in df.columns if col.isdigit()]

df_long = pd.melt(
    df,
    id_vars=["Country Name", "Country Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Agriculture GDP %"
)

df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce")
df_long["Agriculture GDP %"] = pd.to_numeric(df_long["Agriculture GDP %"], errors="coerce")

df_long = df_long.dropna(subset=["Year", "Agriculture GDP %"])

df_long.head()

,Country Name,Country Code,Year,Agriculture GDP %
18,Benin,BEN,1960,46.157723
19,Burkina Faso,BFA,1960,38.484533
20,Bangladesh,BGD,1960,57.474312
29,Brazil,BRA,1960,0.000000
33,Botswana,BWA,1960,43.222960


In [23]:
country_name = "Pakistan"

country_df = df_long[df_long["Country Name"] == country_name].copy()
country_df = country_df.sort_values("Year")

country_df.head()

,Country Name,Country Code,Year,Agriculture GDP %
184,Pakistan,PAK,1960,43.189201
450,Pakistan,PAK,1961,41.727426
716,Pakistan,PAK,1962,40.029233
982,Pakistan,PAK,1963,38.840015
1248,Pakistan,PAK,1964,38.324054


In [25]:
X = country_df[["Year"]]
y = country_df["Agriculture GDP %"]

In [27]:
model = LinearRegression()
model.fit(X, y)

LinearRegression()

In [29]:
country_df["Predicted Agriculture GDP %"] = model.predict(X)

In [31]:
future_years = pd.DataFrame({
    "Year": list(range(2025, 2031))
})

future_predictions = model.predict(future_years)

future_df = pd.DataFrame({
    "Country Name": country_name,
    "Year": future_years["Year"],
    "Agriculture GDP %": np.nan,
    "Predicted Agriculture GDP %": future_predictions,
    "Data Type": "Forecast"
})

In [33]:
actual_df = country_df[["Country Name", "Year", "Agriculture GDP %", "Predicted Agriculture GDP %"]].copy()
actual_df["Data Type"] = "Actual"

In [35]:
forecast_final = pd.concat([actual_df, future_df], ignore_index=True)

forecast_final.head()

,Country Name,Year,Agriculture GDP %,Predicted Agriculture GDP %,Data Type
0,Pakistan,1960,43.189201,35.227486,Actual
1,Pakistan,1961,41.727426,34.962990,Actual
2,Pakistan,1962,40.029233,34.698494,Actual
3,Pakistan,1963,38.840015,34.433997,Actual
4,Pakistan,1964,38.324054,34.169501,Actual


In [37]:
mae = mean_absolute_error(
    country_df["Agriculture GDP %"],
    country_df["Predicted Agriculture GDP %"]
)

rmse = np.sqrt(mean_squared_error(
    country_df["Agriculture GDP %"],
    country_df["Predicted Agriculture GDP %"]
))

r2 = r2_score(
    country_df["Agriculture GDP %"],
    country_df["Predicted Agriculture GDP %"]
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 2.4786012329569136
RMSE: 2.970712717952649
R2 Score: 0.7361731275073149


In [39]:
metrics_df = pd.DataFrame({
    "Country Name": [country_name],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2 Score": [r2]
})

metrics_df

,Country Name,MAE,RMSE,R2 Score
0,Pakistan,2.478601,2.970713,0.736173


In [46]:
forecast_final.to_csv("world population_forecast.csv", index=False)
metrics_df.to_csv("forecast_metrics.csv", index=False)

In [48]:
forecast_final.to_csv(
    r"C:\Users\Hp\Desktop\world population_forecast.csv",
    index=False
)

metrics_df.to_csv(
    r"C:\Users\Hp\Desktop\forecast_metrics.csv",
    index=False
)

In [52]:
forecast_final.to_csv(
    r"C:\Users\Hp\Desktop\world_population_forecast.csv",
    index=False
)

print("Forecast file saved successfully!")

Forecast file saved successfully!
